In [4]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


In [5]:

df = pd.read_csv('./data/annotated_filtered_col.CG_2.fast.tsv', sep='\t')      # or read_parquet / feather …
df

,cluster,chr,start,end,score,c,t,n,flag_euc_gene,flag_het_gene,flag_euc_TE,flag_het_TE,score_masked
0,0,1,101,200,0.8951,350,41,6,False,False,False,False,0.8951
1,0,1,301,400,0.5487,62,51,2,False,False,False,False,0.5487
2,0,1,401,500,0.8246,47,10,1,False,False,False,False,0.8246
3,0,1,501,600,0.7206,98,38,3,False,False,False,False,0.7206
4,0,1,601,700,0.8982,203,23,6,False,False,False,False,0.8982
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5925984,16,5,26974801,26974900,0.8462,22,4,7,False,False,False,False,0.8462
5925985,16,5,26974901,26975000,0.7500,3,1,2,False,False,False,False,NaN
5925986,16,5,26975101,26975200,1.0000,4,0,4,False,False,False,False,NaN
5925987,16,5,26975201,26975300,1.0000,8,0,8,False,False,False,False,1.0000


In [8]:
import numpy as np, pandas as pd
from scipy.special import logit, expit
from scipy.stats import chi2

def drop_uniformly_low(df):
    # Keep windows where ANY cluster survived the 0.2 rule
    g = df.groupby(['chr','start','end'])['score_masked'].apply(lambda s: s.notna().any())
    keep = g[g].reset_index()[['chr','start','end']]
    return df.merge(keep, on=['chr','start','end'], how='inner')

def assign_primary_category(df):
    # Example priority; replace with your overlap-based winner if you have it
    prio = ['flag_euc_gene','flag_het_gene','flag_euc_TE','flag_het_TE']
    catmap = {'flag_euc_gene':'euc_gene','flag_het_gene':'het_gene',
              'flag_euc_TE':'euc_te','flag_het_TE':'het_te'}
    def pick(row):
        for f in prio:
            if row[f]: return catmap[f]
        return np.nan
    df = df.copy()
    df['category'] = df.apply(pick, axis=1)
    # collapse to window-level category (most common or any True)
    wcat = df.groupby(['chr','start','end'])['category'].agg(lambda x: x.dropna().mode()[0] if x.dropna().size else np.nan)
    return df.merge(wcat.rename('win_category'), on=['chr','start','end'], how='left')

def fit_offsets(df_cat):
    agg = df_cat.groupby('cluster')[['c','t']].sum()
    M = agg.sum(axis=1); p = agg['c']/M
    d = logit(np.clip(p,1e-6,1-1e-6))
    # center by M-weighted mean
    return d - np.average(d, weights=M)

def window_stats(df_cat, deltas, tau=20.0, eps=1e-6):
    out = []
    # category prior for EB shrinkage
    agg = df_cat.groupby('cluster')[['c','t']].sum()
    mu = agg['c'].sum() / agg.sum(axis=1).sum()
    a0, b0 = mu*tau, (1-mu)*tau

    for (chr_,start,end), g in df_cat.groupby(['chr','start','end']):
        c = g['c'].to_numpy(); t = g['t'].to_numpy(); m = c + t
        k = g['cluster'].to_numpy()
        keep = m > 0
        if keep.sum() < 2:  # need at least 2 clusters with coverage
            continue
        c, m, k = c[keep], m[keep], k[keep]
        pbar = c.sum()/m.sum()
        p0 = expit(logit(np.clip(pbar,1e-6,1-1e-6)) + deltas.loc[k].to_numpy())
        E = m*p0
        Var = m*p0*(1-p0) + eps
        X2 = ((c - E)**2 / Var).sum()
        dfree = len(c) - 1
        # EB effect size
        p_tilde = (c + a0) / (m + a0 + b0)
        dmax = float(p_tilde.max() - p_tilde.min())
        hi = int(k[p_tilde.argmax()]); lo = int(k[p_tilde.argmin()])
        out.append((chr_,start,end,X2,dfree,dmax,hi,lo))
    res = pd.DataFrame(out, columns=['chr','start','end','X2','df','delta_max','hi_cluster','lo_cluster'])
    # robust dispersion per category
    phi = np.median(res['X2']/np.maximum(res['df'],1)) if len(res) else 1.0
    res['pval'] = 1 - chi2.cdf(res['X2']/max(phi,1e-6), res['df'])
    res['phi'] = phi
    return res

def bh_fdr(p):
    if len(p)==0: return p
    r = np.argsort(p); ranks = np.empty_like(r); ranks[r] = np.arange(1,len(p)+1)
    q = p * len(p) / np.maximum(ranks,1)
    q_sorted = np.minimum.accumulate(np.sort(q)[::-1])[::-1]
    out = np.empty_like(q_sorted); out[r] = q_sorted
    return np.clip(out,0,1)

# ----- run -----
df1 = drop_uniformly_low(df)  # uses your score_masked
df1 = assign_primary_category(df1)

results = []
for cat in ['euc_gene','het_gene','euc_te','het_te']:
    sub = df1[df1['win_category']==cat]
    if sub.empty: continue
    deltas = fit_offsets(sub)
    stats = window_stats(sub, deltas, tau=20.0)
    stats['category'] = cat
    stats['qval'] = bh_fdr(stats['pval'].to_numpy())
    results.append(stats)

dmw = pd.concat(results, ignore_index=True) if results else pd.DataFrame()
# Example calling rule


KeyboardInterrupt: 

In [10]:
dmw

,chr,start,end,X2,df,delta_max,hi_cluster,lo_cluster,pval,phi,category,qval
0,1,5401,5500,13.760077,15,0.141937,11,1,0.879921,1.536786,euc_gene,1.000000
1,1,7001,7100,55.949639,16,0.298853,14,9,0.002540,1.536786,euc_gene,0.023845
2,1,11201,11300,77.019910,16,0.443047,15,0,0.000022,1.536786,euc_gene,0.000418
3,1,23901,24000,19.171355,15,0.243401,3,4,0.642780,1.536786,euc_gene,1.000000
4,1,24001,24100,21.045295,15,0.197810,14,2,0.548825,1.536786,euc_gene,0.984130
...,...,...,...,...,...,...,...,...,...,...,...,...
304598,5,15299801,15299900,18.546799,16,0.197922,12,1,0.743535,1.544736,het_te,1.000000
304599,5,15299901,15300000,13.166474,16,0.189137,5,16,0.931736,1.544736,het_te,1.000000
304600,5,15300001,15300100,14.476215,16,0.200705,7,1,0.897338,1.544736,het_te,1.000000
304601,5,15300201,15300300,16.898293,16,0.133333,13,8,0.813218,1.544736,het_te,1.000000


In [28]:
# write in most effiecient data size
dmw.to_pickle("./data/dmw.pkl")

# # later, reload with the exact same dtypes preserved:
# dmw = pd.read_pickle("./data/dmw.pkl")

In [19]:
calls = dmw[(dmw['qval'] <= 0.05) & (dmw['delta_max'] >= 0.2)& (dmw['pval'] <=0.01)]


In [20]:
calls

,chr,start,end,X2,df,delta_max,hi_cluster,lo_cluster,pval,phi,category,qval
1,1,7001,7100,55.949639,16,0.298853,14,9,0.002540,1.536786,euc_gene,0.023845
2,1,11201,11300,77.019910,16,0.443047,15,0,0.000022,1.536786,euc_gene,0.000418
7,1,24301,24400,63.303077,16,0.323758,0,15,0.000520,1.536786,euc_gene,0.006430
20,1,25801,25900,68.401063,16,0.283734,0,16,0.000165,1.536786,euc_gene,0.002425
27,1,26601,26700,54.791427,16,0.340057,0,12,0.003232,1.536786,euc_gene,0.029005
...,...,...,...,...,...,...,...,...,...,...,...,...
304533,5,15243701,15243800,61.230440,16,0.289502,3,9,0.000879,1.544736,het_te,0.021822
304540,5,15259701,15259800,70.739222,16,0.413612,8,6,0.000105,1.544736,het_te,0.004410
304541,5,15259801,15259900,67.105317,16,0.465839,5,4,0.000240,1.544736,het_te,0.008304
304545,5,15260301,15260400,53.435277,14,0.434845,9,6,0.001689,1.544736,het_te,0.035031


**chr, start, end** — the genomic window (your coords are 1-based inclusive).

**category** — which baseline this window was judged against (e.g., euc_gene). This matters because each category has its own typical methylation level across clusters.

**X2** — the Pearson chi-square statistic for this window, comparing observed methylated counts in each cluster to what we’d expect given the category/feature’s cluster offsets (i.e., the usual pattern for that category). Bigger = more deviation from the baseline pattern.

**df** — degrees of freedom = (#clusters with coverage in this window) − 1.
You’ve got 16 here, which means 17 clusters contributed (e.g., cluster IDs 0–16). If you truly have 16 clusters, two possibilities: either indexing is 0–16 (17 total) or one extra cluster slipped in—worth a quick check.

**phi** — the overdispersion estimate for the whole category (constant within a category block).
phi ≈ 1.54 means counts vary ~1.54× more than simple binomial would predict; we divide X2 by phi before computing the p-value so we don’t overcall.

**pval** — p-value from a χ²(df) test using X2 / phi. This tests “does this window’s across-cluster profile deviate from the category trend?”

**qval** — BH-FDR adjusted p-value within this category (so you can compare windows fairly inside e.g. euc_gene).

**delta_max** — EB-shrunken effect size for this window: the max difference in methylation fraction between any two clusters after shrinking low-coverage estimates toward the category mean. It’s on the same 0–1 scale as methylation fraction (so 0.30 ≈ 30 percentage points).

**hi_cluster / lo_cluster** — which clusters achieve that delta_max contrast (highest vs lowest shrunken methylation for this window).

---

***How to interpret a row (example)***
chr1:11201-11300 (euc_gene):

X2=77.02, df=16, phi=1.5368 → strong deviation from the expected pattern for euc_gene windows.

pval=2.2e-5, qval=4.18e-4 → significant after FDR within euc_gene.

delta_max=0.443, hi_cluster=15, lo_cluster=0 → after EB shrinkage, cluster 15 is ~44 percentage points more methylated than cluster 0 at this window. That’s a big effect.

A row with df=14 (e.g., 15260301–15260400) just means 2 clusters had zero coverage and were excluded from the test for that window.

In [16]:
calls['category'].unique()

array(['euc_gene', 'het_gene', 'euc_te', 'het_te'], dtype=object)

In [18]:
dmw[dmw["start"] > 21327200]

,chr,start,end,X2,df,delta_max,hi_cluster,lo_cluster,pval,phi,category,qval
24596,1,21328401,21328500,81.411348,16,0.416370,3,10,0.000008,1.536786,euc_gene,0.000164
24597,1,21328501,21328600,25.815812,16,0.280582,3,12,0.398746,1.536786,euc_gene,0.853643
24598,1,21328601,21328700,23.749664,16,0.243632,8,4,0.491663,1.536786,euc_gene,0.939223
24599,1,21328701,21328800,30.171812,16,0.332300,16,1,0.237204,1.536786,euc_gene,0.648239
24600,1,21328901,21329000,50.699843,16,0.283705,3,10,0.007411,1.536786,euc_gene,0.056485
...,...,...,...,...,...,...,...,...,...,...,...,...
187830,5,26893101,26893200,78.143690,16,0.432695,5,10,0.000015,1.527399,euc_te,0.000556
187831,5,26893201,26893300,24.144295,12,0.165110,2,12,0.200215,1.527399,euc_te,0.648012
187832,5,26893301,26893400,45.335038,14,0.226978,6,16,0.008441,1.527399,euc_te,0.085656
187833,5,26893401,26893500,16.558965,15,0.183134,2,6,0.763762,1.527399,euc_te,1.000000


In [21]:
def explain_missing(df, dmw, chr_, start, end):
    key = (chr_, start, end)
    w = df[(df['chr']==chr_) & (df['start']==start) & (df['end']==end)]
    if w.empty:
        return "Not in raw df (check chr/start/end types)."

    # 1) Mask
    all_masked = w['score_masked'].isna().all()
    # 2) Coverage across clusters
    m = (w['c'] + w['t']).to_numpy()
    n_cov = (m > 0).sum()
    # 3) Category
    cat_cols = ['flag_euc_gene','flag_het_gene','flag_euc_TE','flag_het_TE']
    has_any_flag = w[cat_cols].any(axis=None)
    # 4) Did it appear in dmw at all?
    in_dmw = not dmw[(dmw['chr']==chr_) & (dmw['start']==start) & (dmw['end']==end)].empty

    return {
        "present_in_raw": True,
        "all_score_masked_nan": bool(all_masked),
        "clusters_with_coverage": int(n_cov),
        "has_any_category_flag": bool(has_any_flag),
        "present_in_dmw": bool(in_dmw)
    }

# Example:
explain_missing(df, dmw, 1, 101, 200)


{'present_in_raw': True,
 'all_score_masked_nan': False,
 'clusters_with_coverage': 17,
 'has_any_category_flag': False,
 'present_in_dmw': False}

In [26]:
dmw[(dmw['pval']<0.000000005)]

,chr,start,end,X2,df,delta_max,hi_cluster,lo_cluster,pval,phi,category,qval
83,1,74701,74800,138.492929,16,0.516003,12,1,2.379097e-12,1.536786,euc_gene,1.719925e-10
84,1,76801,76900,115.743048,14,0.498802,2,4,2.071502e-10,1.536786,euc_gene,1.109389e-08
85,1,77801,77900,283.205748,16,0.509852,16,0,0.000000e+00,1.536786,euc_gene,0.000000e+00
125,1,117201,117300,209.579299,16,0.469321,16,0,0.000000e+00,1.536786,euc_gene,0.000000e+00
129,1,121701,121800,121.351245,16,0.345870,4,16,2.556039e-10,1.536786,euc_gene,1.349449e-08
...,...,...,...,...,...,...,...,...,...,...,...,...
304310,5,15121101,15121200,181.250987,16,0.705760,15,1,0.000000e+00,1.544736,het_te,0.000000e+00
304346,5,15144001,15144100,121.394279,16,0.326539,0,9,2.989358e-10,1.544736,het_te,9.511207e-08
304384,5,15152101,15152200,142.066931,16,0.548679,14,7,1.083911e-12,1.544736,het_te,5.970099e-10
304398,5,15160601,15160700,113.592816,15,0.475163,5,13,1.039966e-09,1.544736,het_te,2.933207e-07


In [27]:
df[df['start']==77801]

,cluster,chr,start,end,score,c,t,n,flag_euc_gene,flag_het_gene,flag_euc_TE,flag_het_TE,score_masked
126,0,1,77801,77900,0.0028,1,354,8,True,False,False,False,0.0028
354488,1,1,77801,77900,0.0034,1,296,8,True,False,False,False,0.0034
708825,2,1,77801,77900,0.0072,1,138,8,True,False,False,False,0.0072
1062780,3,1,77801,77900,0.0000,0,75,8,True,False,False,False,0.0000
1416416,4,1,77801,77900,0.0000,0,127,8,True,False,False,False,0.0000
1770467,5,1,77801,77900,0.0000,0,63,8,True,False,False,False,0.0000
2123599,6,1,77801,77900,0.0000,0,131,8,True,False,False,False,0.0000
2477422,7,1,77801,77900,0.0000,0,101,8,True,False,False,False,0.0000
2831157,8,1,77801,77900,0.0000,0,57,8,True,False,False,False,0.0000
3182968,9,1,77801,77900,0.0000,0,18,7,True,False,False,False,0.0000


In [6]:
import numpy as np, pandas as pd
from scipy.special import logit, expit
from scipy.stats import chi2

# --- helpers ---------------------------------------------------------------

def bh_fdr(p):
    p = np.asarray(p)
    if p.size == 0: return p
    order = np.argsort(p)
    ranks = np.empty_like(order); ranks[order] = np.arange(1, p.size+1)
    q = p * p.size / ranks
    q_sorted = np.minimum.accumulate(np.sort(q)[::-1])[::-1]
    out = np.empty_like(q_sorted); out[order] = q_sorted
    return np.clip(out, 0, 1)

def fit_offsets(df_cat):
    """Category-level cluster offsets (logit) with M-weighted centering."""
    agg = df_cat.groupby('cluster')[['c','t']].sum()
    M = agg.sum(axis=1)
    p = (agg['c'] / M).clip(1e-6, 1-1e-6)
    d = logit(p)
    return d - np.average(d, weights=M)

# --- core per-window stats (adds influence + trimmed delta) ---------------

def window_stats_robust(
    df_cat, deltas, tau=20.0, eps=1e-6
):
    """
    Returns per-window stats including X2, df, EB deltas, influence, LOCO (no phi yet),
    and coverage of hi/lo clusters.
    """
    out = []

    # EB prior (per-category)
    agg = df_cat.groupby('cluster')[['c','t']].sum()
    mu = agg['c'].sum() / agg.sum(axis=1).sum()
    a0, b0 = mu*tau, (1-mu)*tau

    # Process each window
    for (chr_, start, end), g in df_cat.groupby(['chr','start','end'], sort=False):
        c = g['c'].to_numpy(); t = g['t'].to_numpy(); m = c + t
        k = g['cluster'].to_numpy()
        keep = m > 0
        if keep.sum() < 2:
            # not enough clusters with coverage to test
            out.append((chr_,start,end, np.nan, 0, 0.0, -1, -1, 0.0, -1, 0, 0, 0.0, 0.0))
            continue

        c, m, k = c[keep], m[keep], k[keep]

        # expected under null with offsets
        pbar = c.sum() / m.sum()
        p0   = expit(logit(np.clip(pbar,1e-6,1-1e-6)) + deltas.loc[k].to_numpy())
        E    = m * p0
        Var  = m * p0 * (1 - p0) + eps

        # Pearson X^2 and contributions
        contrib = ( (c - E)**2 / Var )
        X2 = contrib.sum()
        dfree = len(c) - 1
        top_idx = int(np.argmax(contrib))
        top_frac = float(contrib[top_idx] / max(X2, 1e-12))
        top_cluster = int(k[top_idx])
        top_cov = int(m[top_idx])

        # LOCO test: drop top contributor, recompute X2 with re-fitted pbar
        mask = np.ones(len(c), dtype=bool); mask[top_idx] = False
        if mask.sum() >= 2:
            pbar_loco = c[mask].sum() / m[mask].sum()
            p0_loco   = expit(logit(np.clip(pbar_loco,1e-6,1-1e-6)) + deltas.loc[k[mask]].to_numpy())
            E_loco    = m[mask] * p0_loco
            Var_loco  = m[mask] * p0_loco * (1 - p0_loco) + eps
            X2_loco   = ((c[mask] - E_loco)**2 / Var_loco).sum()
            df_loco   = int(mask.sum() - 1)
        else:
            X2_loco, df_loco = np.nan, 0

        # EB shrunken cluster proportions
        p_tilde = (c + a0) / (m + a0 + b0)
        # classic and trimmed deltas
        hi_i = int(np.argmax(p_tilde)); lo_i = int(np.argmin(p_tilde))
        delta_max = float(p_tilde[hi_i] - p_tilde[lo_i])
        hi_cluster = int(k[hi_i]); lo_cluster = int(k[lo_i])
        m_hi = int(m[hi_i]);       m_lo      = int(m[lo_i])

        # trimmed delta (P90 - P10) among clusters with coverage
        p_sorted = np.sort(p_tilde)
        p10 = float(np.percentile(p_sorted, 10))
        p90 = float(np.percentile(p_sorted, 90))
        delta_trim = float(p90 - p10)

        out.append((chr_, start, end,
                    X2, dfree,
                    delta_max, hi_cluster, lo_cluster,
                    delta_trim, top_cluster, top_cov, hi_cluster, lo_cluster, m_hi, m_lo,
                    X2_loco, df_loco))

    cols = ['chr','start','end',
            'X2','df',
            'delta_max','hi_cluster','lo_cluster',
            'delta_max_trim','top_cluster','top_cov','hi_cluster_dup','lo_cluster_dup','m_hi','m_lo',
            'X2_loco','df_loco']
    res = pd.DataFrame(out, columns=cols)
    # drop the dup helper cols
    res = res.drop(columns=['hi_cluster_dup','lo_cluster_dup'])
    return res

# --- per-category wrapper that computes phi, pvals, qvals -----------------

def dmw_with_robust_guards(
    df_cat,
    tau=20.0,
    q_alpha=0.05,
    delta_trim_min=0.20,
    neighbor_delta_min=0.15,
    require_neighbors=2,
    top_contrib_max=0.80,
    coverage_floor_hi=10,
    coverage_floor_lo=10,
    eps=1e-6
):
    """
    Runs per-window stats for one category, then applies robust calling rules.
    Returns (stats_df, calls_df).
    """
    if df_cat.empty:
        return (pd.DataFrame(), pd.DataFrame())

    # Offsets
    deltas = fit_offsets(df_cat)

    # Per-window stats (no phi/p yet)
    stats = window_stats_robust(df_cat, deltas, tau=tau, eps=eps)

    # Robust dispersion (phi) and p-values for X2 and LOCO X2
    ok = stats['df'] > 0
    phi = np.median((stats.loc[ok, 'X2'] / stats.loc[ok, 'df']).replace([np.inf, -np.inf], np.nan).dropna()) if ok.any() else 1.0
    phi = max(phi, 1e-6)
    stats['phi']  = phi
    stats['pval'] = 1 - chi2.cdf(stats['X2'] / phi, stats['df'].clip(lower=1))
    # LOCO p
    loco_ok = stats['df_loco'] > 0
    stats['p_loco'] = np.nan
    stats.loc[loco_ok, 'p_loco'] = 1 - chi2.cdf(stats.loc[loco_ok, 'X2_loco'] / phi,
                                               stats.loc[loco_ok, 'df_loco'])

    # FDR within category
    stats['qval'] = np.nan
    stats.loc[ok, 'qval'] = bh_fdr(stats.loc[ok, 'pval'].to_numpy())

    # Neighbor support: count within ±1 adjacent bin on same chr with delta_max_trim ≥ neighbor_delta_min
    stats = stats.sort_values(['chr','start','end']).reset_index(drop=True)
    stats['neighbor_support'] = 0
    for chr_, sub_idx in stats.groupby('chr').groups.items():
        idx = np.array(sorted(sub_idx))
        s   = stats.loc[idx, 'start'].to_numpy()
        e   = stats.loc[idx, 'end'].to_numpy()
        dlt = stats.loc[idx, 'delta_max_trim'].to_numpy()
        cond = dlt >= neighbor_delta_min
        # previous neighbor within one bin?
        prev_ok = np.zeros_like(cond, dtype=int)
        prev_ok[1:] = (cond[:-1] & (s[1:] <= e[:-1] + (e[:-1]-s[:-1]+1)))
        # next neighbor within one bin?
        next_ok = np.zeros_like(cond, dtype=int)
        next_ok[:-1] = (cond[1:] & (s[1:] <= e[:-1] + (e[:-1]-s[:-1]+1)))
        stats.loc[idx, 'neighbor_support'] = prev_ok + next_ok

    # Robust call rule
    def reason_row(r):
        if not np.isfinite(r['qval']) or r['qval'] > q_alpha: return 'ns_qval'
        if r['delta_max_trim'] < delta_trim_min:              return 'weak_effect'
        top_frac = (r['X2'] and (r['X2']>0)) and ((r['X2'] - (r['X2'] - r['X2'])) or 0)  # dummy to keep vars defined
        top_frac = np.nan if not np.isfinite(r['X2']) else (np.nan if r['X2']==0 else
                        ((r['X2'] - (r['X2'] - 0)) / r['X2']))  # will be overwritten below anyway

        # compute top contribution fraction safely
        top_frac = np.nan
        if np.isfinite(r['X2']) and r['X2'] > 0:
            # We didn't store top_contrib value explicitly, but we can reconstruct from p_loco and X2 if needed.
            # Simpler: treat "single-cluster dominance" by using coverage floor checks and p_loco.
            top_frac = np.nan  # left blank; p_loco + coverage floors do the heavy lifting.

        if ( (r['m_hi'] < coverage_floor_hi) or (r['m_lo'] < coverage_floor_lo) ) and ( (r['p_loco'] is np.nan) or (r['p_loco'] > q_alpha) ):
            return 'low_cov_hi_lo_and_loco_ns'
        if r['neighbor_support'] < require_neighbors:         return 'no_neighbor_support'
        return 'ok'

    # we’ll also compute a simpler dominance flag using LOCO directly:
    stats['dominance_blocked'] = (stats['p_loco'].notna()) & (stats['p_loco'] > q_alpha)

    stats['reason'] = stats.apply(reason_row, axis=1)
    calls = stats[stats['reason'] == 'ok'].copy()

    return stats, calls


In [30]:
# 1) Drop uniformly-low windows (all clusters score_masked NaN) and assign primary category
# (reuse the assign_primary_category() from earlier, or plug in your “winner by bp overlap”.)
df1 = drop_uniformly_low(df)
df1 = assign_primary_category(df1)   # adds df1['win_category']

# 2) Run per-category and concatenate
results = []
calls_all = []
for cat in ['euc_gene','het_gene','euc_te','het_te']:
    sub = df1[df1['win_category'] == cat]
    stats, calls = dmw_with_robust_guards(
        sub,
        tau=20.0,
        q_alpha=0.05,
        delta_trim_min=0.20,
        neighbor_delta_min=0.15,
        require_neighbors=1,      # set to 2 for stricter “2-of-3 bins” support
        coverage_floor_hi=10,
        coverage_floor_lo=10
    )
    if not stats.empty:
        stats['category'] = cat
        results.append(stats)
        calls['category'] = cat
        calls_all.append(calls)

dmw_robust = pd.concat(results, ignore_index=True) if results else pd.DataFrame()
calls_robust = pd.concat(calls_all, ignore_index=True) if calls_all else pd.DataFrame()


In [33]:
dmw_robust.to_pickle("./data/dmw_robust.pkl")

In [32]:
calls_robust

,chr,start,end,X2,df,delta_max,hi_cluster,lo_cluster,delta_max_trim,top_cluster,...,X2_loco,df_loco,phi,pval,p_loco,qval,neighbor_support,dominance_blocked,reason,category
0,1,24301,24400,63.303077,16,0.323758,0,15,0.235239,15,...,33.888008,15.0,1.536786,5.201946e-04,1.064678e-01,6.429773e-03,1,True,ok,euc_gene
1,1,25801,25900,68.401063,16,0.283734,0,16,0.233424,9,...,58.003879,15.0,1.536786,1.649916e-04,9.844080e-04,2.425366e-03,2,False,ok,euc_gene
2,1,26601,26700,54.791427,16,0.340057,0,12,0.204524,12,...,27.864254,15.0,1.536786,3.231771e-03,2.558146e-01,2.900470e-02,2,True,ok,euc_gene
3,1,117101,117200,95.418897,16,0.454657,15,0,0.324092,15,...,69.541184,15.0,1.536786,2.317766e-07,6.989049e-05,7.101292e-06,1,False,ok,euc_gene
4,1,117201,117300,209.579299,16,0.469321,16,0,0.315957,16,...,130.321252,15.0,1.536786,0.000000e+00,9.155010e-12,0.000000e+00,1,False,ok,euc_gene
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7202,5,15152401,15152500,76.102981,16,0.345352,15,2,0.291096,0,...,62.357804,15.0,1.544736,2.998746e-05,3.989609e-04,1.631343e-03,1,False,ok,het_te
7203,5,15182201,15182300,124.173308,16,0.394110,16,7,0.264723,7,...,102.892617,15.0,1.544736,1.418309e-10,1.782187e-08,4.899795e-08,1,False,ok,het_te
7204,5,15203801,15203900,73.960816,16,0.356692,15,10,0.289099,10,...,61.158534,15.0,1.544736,4.961880e-05,5.225181e-04,2.429303e-03,1,False,ok,het_te
7205,5,15259701,15259800,70.739222,16,0.413612,8,6,0.340276,6,...,58.261694,15.0,1.544736,1.047749e-04,9.935832e-04,4.410364e-03,1,False,ok,het_te


In [34]:
calls_robust.to_pickle("./data/calls_robust.pkl")

In [37]:
df[(df['start']==21328401) & (df['chr']==1)]	

,cluster,chr,start,end,score,c,t,n,flag_euc_gene,flag_het_gene,flag_euc_TE,flag_het_TE,score_masked
61558,0,1,21328401,21328500,0.7649,257,79,8,True,False,False,False,0.7649
415910,1,1,21328401,21328500,0.6847,278,128,8,True,False,False,False,0.6847
770164,2,1,21328401,21328500,0.8054,120,29,8,True,False,False,False,0.8054
1124039,3,1,21328401,21328500,0.8770,107,15,8,True,False,False,False,0.8770
1477777,4,1,21328401,21328500,0.7664,105,32,8,True,False,False,False,0.7664
1831620,5,1,21328401,21328500,0.8846,69,9,8,True,False,False,False,0.8846
2184900,6,1,21328401,21328500,0.6887,73,33,8,True,False,False,False,0.6887
2538727,7,1,21328401,21328500,0.8305,98,20,8,True,False,False,False,0.8305
2892070,8,1,21328401,21328500,0.7436,58,20,8,True,False,False,False,0.7436
3243535,9,1,21328401,21328500,0.5526,21,17,8,True,False,False,False,0.5526
